In [7]:
from __future__ import annotations

import asyncio
import json
import os
import time
from typing import Any, Dict, List, Optional
from dotenv import load_dotenv
import httpx
import asyncio
import ollama

load_dotenv('backend/.env')
class OllamaClient:
    def __init__(self, host: Optional[str] = None, model: Optional[str] = None, timeout_s: Optional[float] = None):
        self.host = host or os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
        self.model = model or os.getenv("OLLAMA_MODEL", "gemma4:e4b").split(",")[0].strip()
        self.timeout_s = timeout_s or float(os.getenv("OLLAMA_TIMEOUT_S", "300"))
        # Use ollama.AsyncClient instead of AsyncClient
        self.client = ollama.AsyncClient(self.host, timeout=httpx.Timeout(self.timeout_s))
        self.api_key = os.getenv("OLLAMA_API_KEY")
        return None

client = OllamaClient()

async def chat() -> Any:
    conversation = [
        {"role": "system", "content": "You are a helpful assistant and nfl data analyst."}
    ]

    user_prompt = input("Enter your message: ")
    conversation.append({"role": "user", "content": user_prompt})

    async for part in await client.client.chat(model=client.model, messages=conversation, stream=True):
        print(part.message.content, end="", flush=True)

    if input("\nContinue? (y/n): ").lower() != "y":
        print("Chat ended.")

try:
    loop = asyncio.get_running_loop()
    await chat()
except RuntimeError:
    asyncio.run(chat())

CancelledError: 